Load the dataset

In [1]:
import pandas as pd
meta_qa = pd.read_parquet("parquet_data/qa_nl.parquet")

In [2]:
import pandas as pd
outp = pd.read_parquet("output/masterRAG/rag_outputs.parquet")
outp

,query_id,query,answer,retrieved_ids,latency_sec
0,746,Ik ben gedagvaard. Wat is een dagvaarding?,Kort gezegd is een dagvaarding in het Belgisch...,"[15044, 15376, 25547, 4684, 14700, 4789, 15394...",25.244993
1,1768,Wie oefent het ouderlijk gezag uit als één van...,Kort antwoord: alleen die ouder die door de fa...,"[3791, 3812, 3797, 15319, 3790, 3796, 3721, 38...",34.193376
2,305,Moet ik branddetectors installeren in Brussel?,Kort antwoord: ja. Volgens de Brusselse huisve...,"[23173, 2246, 2247, 13617, 23185, 4147, 2376, ...",30.239648
3,1811,"Als ik mijn consumentenkrediet niet betaal, ku...",Ja. In principe kunnen uw inkomsten voor besla...,"[15646, 15632, 14614, 25394, 9505, 6044, 15625...",25.595162
4,1558,Wat te doen als er een nieuwe huisgenoot in Br...,Kort: geef de komst van een nieuwe huisgenoot ...,"[4108, 4147, 4129, 4142, 4114, 1681, 2545, 152...",35.535536
...,...,...,...,...,...
1430,1106,De bal van mijn kind is in de tuin van mijn bu...,Kort antwoord: ja — als de bal per ongeluk op ...,"[5357, 22685, 4810, 2715, 4804, 7018, 4517, 21...",25.864287
1431,134,Hoe lang moet men wachten voordat men een besl...,Uiterlijk binnen 30 dagen na ontvangst van de ...,"[26674, 26678, 26676, 26673, 7674, 26744, 2201...",17.992402
1432,1031,Ik woon samen met mijn partner en/of mijn kind...,Kort antwoord: de woonplaats (domicilie) van e...,"[3820, 3791, 14608, 25611, 3497, 3801, 3806, 2...",19.578389
1433,1715,Welke documenten en informatie mag het OCMW ei...,Kort: het OCMW mag van een aanvrager “alle voo...,"[26674, 26678, 26673, 26955, 7969, 588, 22019,...",21.339245


In [3]:
#pip install deepeval

In [4]:
from deepeval import evaluate
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
from deepeval.metrics.g_eval import Rubric
from tqdm import tqdm

goldens = [Golden(input=question) for question in meta_qa.question]
dataset = EvaluationDataset(goldens=goldens)

for i, golden in enumerate(tqdm(dataset.goldens, desc="Building test cases")):
    test_case = LLMTestCase(
        input=meta_qa.question[i],
        actual_output=outp.answer[i],
        expected_output=meta_qa.answer[i]
    )
    dataset.add_test_case(test_case)

Building test cases: 100%|██████████| 1435/1435 [00:00<00:00, 5032.92it/s]


In [5]:
from deepeval.metrics import GEval
from deepeval.metrics.g_eval import Rubric
from deepeval.test_case import LLMTestCase, SingleTurnParams
correctness_metric = GEval(
    name="Correctness",
    criteria=(
        "Determine whether the actual_output is either an exact match to the "
        "expected_output, or a relevant statement that accurately conveys the "
        "same legal information."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    rubric=[
        Rubric(score_range=(0, 2), expected_outcome="Critical Failure/Incorrect — contradicts or misstates the law relative to the expected_output."),
        Rubric(score_range=(3, 4), expected_outcome="Poor/Significant Omissions — captures only a small fragment of the expected content or omits key legal elements."),
        Rubric(score_range=(5, 6), expected_outcome="Acceptable/Partially Complete — broadly correct but missing secondary details or nuance."),
        Rubric(score_range=(7, 9), expected_outcome="Good/Mostly Accurate — closely tracks the expected_output with only minor imprecision."),
        Rubric(score_range=(10, 10), expected_outcome="Excellent/Semantically Equivalent — fully and accurately conveys the same meaning as the expected_output, regardless of exact wording."),
    ],
    model = "gpt-5-mini-2025-08-07"
)

In [6]:
import math
import time
import pandas as pd
from tqdm import tqdm
from deepeval import evaluate
from deepeval.evaluate import DisplayConfig
from deepeval.evaluate import DisplayConfig, AsyncConfig

chunk_size = 100
max_concurrent = 20
max_retries = 3
base_delay = 2  # seconds, doubles each retry

test_cases = dataset.test_cases
n = len(test_cases)
num_chunks = math.ceil(n / chunk_size)

rows = []
failed_chunks = []

with tqdm(total=n, desc="Evaluating", unit="case") as pbar:
    for i in range(num_chunks):
        start = i * chunk_size
        chunk = test_cases[start:start + chunk_size]

        results = None
        for attempt in range(1, max_retries + 1):
            try:
                results = evaluate(
                    test_cases=chunk,
                    metrics=[correctness_metric],
                    display_config=DisplayConfig(show_indicator=True, print_results=False),
                    async_config=AsyncConfig(run_async=True, max_concurrent=max_concurrent),
                )
                break  # success, exit retry loop
            except Exception as e:
                if attempt == max_retries:
                    tqdm.write(f"Chunk {i} failed after {max_retries} attempts: {e}")
                    failed_chunks.append(i)
                else:
                    delay = base_delay * (2 ** (attempt - 1))
                    tqdm.write(f"Chunk {i} attempt {attempt} failed ({e}), retrying in {delay}s...")
                    time.sleep(delay)

        if results is not None:
            for test_result in results.test_results:
                if not test_result.metrics_data:
                    continue
                for metric in test_result.metrics_data:
                    rows.append({
                        "input": test_result.input,
                        "actual_output": test_result.actual_output,
                        "expected_output": test_result.expected_output,
                        "metric_name": metric.name,
                        "score": metric.score,
                        "reason": metric.reason,
                        "success": metric.success,
                    })

        pbar.update(len(chunk))

df = pd.DataFrame(rows)

Evaluating:   0%|          | 0/1435 [00:00<?, ?case/s]

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-5-mini-2025-08-07, strict=False, 
async_mode=True)...

Output()

⚠ WARNING: No hyperparameters logged.
» ]8;id=6561;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 124.99s | token cost: 0.3034700000000001 USD)
» Test Results (100 total tests):
   » Pass Rate: 48.0% | Passed: 48 | Failed: 52

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

...

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-5-mini-2025-08-07, strict=False, 
async_mode=True)...

Output()

⚠ WARNING: No hyperparameters logged.
» ]8;id=894961;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 74.16s | token cost: 0.10989424999999999 USD)
» Test Results (35 total tests):
   » Pass Rate: 48.57% | Passed: 17 | Failed: 18

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Evaluating: 100%|██████████| 1435/1435 [36:55<00:00,  1.54s/case]


In [7]:
# Save as Parquet
df.to_parquet("output/masterRAG/correctness_results.parquet", engine="pyarrow", index=False)

In [8]:
import pandas as pd
eval = pd.read_parquet("output/masterRAG/correctness_results.parquet")
eval

,input,actual_output,expected_output,metric_name,score,reason,success
0,"Als ik mijn consumentenkrediet niet betaal, ku...",Ja. In principe kunnen uw inkomsten voor besla...,"Ja. Het is geen beslag, maar een cessie van lo...",Correctness [GEval],0.1,The Actual Output contradicts and omits the co...,False
1,Ik ben gedagvaard. Wat is een dagvaarding?,Kort gezegd is een dagvaarding in het Belgisch...,De dagvaarding is het akte waarmee een procedu...,Correctness [GEval],0.4,The response correctly identifies the core poi...,False
2,Ik ben getrouwd. Moet ik een familiale aanspra...,Kort antwoord: Op basis van de aangeleverde we...,"U bent niet verplicht, maar het is aan te rade...",Correctness [GEval],0.6,The Actual Output correctly answers the core l...,True
3,Ik word wettelijk samenwonend. We zijn ingesch...,Kort: Nee — alleen op hetzelfde adres ingeschr...,Niet noodzakelijk. U wordt niet automatisch we...,Correctness [GEval],0.6,The response correctly answers the user’s ques...,True
4,Wat is een beslag op inkomsten?,Een beslag op inkomsten is in essentie het ger...,Voordat u verdergaat\n\nDeze fiche heeft betre...,Correctness [GEval],0.6,Addresses the legal question and correctly cit...,True
...,...,...,...,...,...,...,...
1430,"Herstellingen, onderhoud en werken: wie doet w...",Kort: in Wallonië geldt dat de huurder verantw...,Dit document is meer dan een jaar geleden bijg...,Correctness [GEval],0.4,The actual output correctly states the core le...,False
1431,Hoe lang moet men wachten voordat men een besl...,Uiterlijk binnen 30 dagen na ontvangst van de ...,Het OCMW moet zijn beslissing uiterlijk 30 dag...,Correctness [GEval],0.9,Addresses the input question and core legal ru...,True
1432,"Ik ben student, kan het OCMW mij verplichten o...",Kort: Uit de aangeleverde wetsartikelen kan ni...,Ja. Het OCMW kan u verplichten om een studente...,Correctness [GEval],0.2,The actual output contradicts the expected con...,False
1433,Ik woon samen met mijn partner en/of mijn kind...,Kort antwoord: de woonplaats (domicilie) van e...,Voordat we verder gaan\n\nTot april 2017 konde...,Correctness [GEval],0.5,The Actual Output correctly states the core le...,True


In [9]:
#pip install ragas==0.4.3 langchain==0.3.30 langchain-community==0.3.31 langchain-openai==0.3.35

In [10]:
from retrievers import DocsRetriever
def get_context(article_ids):
    article_tuples = [(id_val, 0) for id_val in article_ids]
    return DocsRetriever(article_tuples).documents()
outp["retrieved_contexts"] = outp["retrieved_ids"].apply(get_context)

In [11]:
outp

,query_id,query,answer,retrieved_ids,latency_sec,retrieved_contexts
0,746,Ik ben gedagvaard. Wat is een dagvaarding?,Kort gezegd is een dagvaarding in het Belgisch...,"[15044, 15376, 25547, 4684, 14700, 4789, 15394...",25.244993,[Art. 1035. Op de dag en het uur bepaald in he...
1,1768,Wie oefent het ouderlijk gezag uit als één van...,Kort antwoord: alleen die ouder die door de fa...,"[3791, 3812, 3797, 15319, 3790, 3796, 3721, 38...",34.193376,[Art. 374. (§ 1.) Wanneer de ouders niet samen...
2,305,Moet ik branddetectors installeren in Brussel?,Kort antwoord: ja. Volgens de Brusselse huisve...,"[23173, 2246, 2247, 13617, 23185, 4147, 2376, ...",30.239648,[Art. 4bis. Elke woning wordt uitgerust met mi...
3,1811,"Als ik mijn consumentenkrediet niet betaal, ku...",Ja. In principe kunnen uw inkomsten voor besla...,"[15646, 15632, 14614, 25394, 9505, 6044, 15625...",25.595162,[Art. 1412quater. § 1er. Onder voorbehoud van ...
4,1558,Wat te doen als er een nieuwe huisgenoot in Br...,Kort: geef de komst van een nieuwe huisgenoot ...,"[4108, 4147, 4129, 4142, 4114, 1681, 2545, 152...",35.535536,[Art. 1716_BRUSSELS_HOOFDSTEDELIJK_GEWEST. [NO...
...,...,...,...,...,...,...
1430,1106,De bal van mijn kind is in de tuin van mijn bu...,Kort antwoord: ja — als de bal per ongeluk op ...,"[5357, 22685, 4810, 2715, 4804, 7018, 4517, 21...",25.864287,[Art. 68.Onverminderd de toepassing van artike...
1431,134,Hoe lang moet men wachten voordat men een besl...,Uiterlijk binnen 30 dagen na ontvangst van de ...,"[26674, 26678, 26676, 26673, 7674, 26744, 2201...",17.992402,[Art. 19.§ 1. Met het oog op de toekenning van...
1432,1031,Ik woon samen met mijn partner en/of mijn kind...,Kort antwoord: de woonplaats (domicilie) van e...,"[3820, 3791, 14608, 25611, 3497, 3801, 3806, 2...",19.578389,[Art. 390. Behoudens hetgeen is bepaald in art...
1433,1715,Welke documenten en informatie mag het OCMW ei...,Kort: het OCMW mag van een aanvrager “alle voo...,"[26674, 26678, 26673, 26955, 7969, 588, 22019,...",21.339245,[Art. 19.§ 1. Met het oog op de toekenning van...


In [12]:
import os
import asyncio
import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness
from tqdm.asyncio import tqdm_asyncio

load_dotenv()

client = AsyncOpenAI(api_key=os.getenv("OPENAI_API"))
llm = llm_factory("gpt-4o-mini", client=client, max_tokens=16384)
scorer = Faithfulness(llm=llm)

CONCURRENCY = 15
TIMEOUT = 90
MAX_RETRIES = 5
BATCH_SIZE = 100

async def score_row(i, row, sem):
    question = row["query"]
    answer = row["answer"]
    context = row["retrieved_contexts"]
    score = None

    async with sem:
        for attempt in range(MAX_RETRIES):
            try:
                result = await asyncio.wait_for(
                    scorer.ascore(
                        user_input=question,
                        response=answer,
                        retrieved_contexts=context,
                    ),
                    timeout=TIMEOUT,
                )
                score = result.value
                break
            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    await asyncio.sleep(min(2 ** attempt, 20))
                else:
                    print(f"Row {i}: failed after {MAX_RETRIES} attempts ({e})")

    return {"question": question, "answer": answer, "faithfulness_score": score}

async def run_evaluation(outp: pd.DataFrame) -> pd.DataFrame:
    sem = asyncio.Semaphore(CONCURRENCY)
    all_results = [] 

    num_batches = (len(outp) + BATCH_SIZE - 1) // BATCH_SIZE

    for i in range(num_batches):
        start = i * BATCH_SIZE
        end = min(start + BATCH_SIZE, len(outp))
        batch_df = outp.iloc[start:end]

        tasks = [score_row(idx, row, sem) for idx, row in batch_df.iterrows()]
        batch_results = await tqdm_asyncio.gather(*tasks, desc=f"Batch {i + 1}/{num_batches}")
        all_results.extend(batch_results)

    results_df = pd.DataFrame(all_results)
    results_df.to_parquet("output/masterRAG/faithfulness_results.parquet", index=False)
    return results_df

# Usage:
results_df = await run_evaluation(outp)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Batch 15/15: 100%|██████████| 35/35 [00:48<00:00,  1.39s/it]
